# PPGCC/UFPI — Projeto de Redes de Computadores 2026-1
## Fase 1: Análise Comparativa TCP vs. R-UDP/GBN

**Aluno:** Arthur Sabino Santos  
**Matrícula:** 20261005029  
**Professor:** Rayner Gomes Sousa

---

## Célula 1 — Instalação de dependências

In [ ]:
# IMPORTANTE: instala kaleido 0.2.1 (versão sem dependência de Chrome)
!pip install -q plotly seaborn pandas matplotlib numpy scapy
!pip install -q "kaleido==0.2.1"
print('Instalação concluída!')

## Célula 2 — Montar o Google Drive

Faça upload da pasta `data/` do projeto para o Google Drive antes de executar.
Ajuste o `BASE_DIR` para o caminho correto.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ⚠️ Ajuste este caminho para onde você fez upload da pasta 'data/'
BASE_DIR = '/content/drive/MyDrive/ppgcc-redes-fase1/data'

import os
LOG_DIR  = os.path.join(BASE_DIR, 'logs')
CSV_DIR  = os.path.join(BASE_DIR, 'csv')
PLOT_DIR = os.path.join(BASE_DIR, 'plots')
os.makedirs(PLOT_DIR, exist_ok=True)
print(f'LOG_DIR  = {LOG_DIR}')
print(f'CSV_DIR  = {CSV_DIR}')
print(f'PLOT_DIR = {PLOT_DIR}')
print('OK!' if os.path.exists(LOG_DIR) else '❌ Pasta não encontrada — verifique BASE_DIR')

## Célula 3 — Importações e carregamento dos dados

In [ ]:
import json, glob
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 150

SCENARIOS = ['A', 'B', 'C']
SCENARIO_LABELS = {'A': 'A (0%/10ms)', 'B': 'B (10%/50ms)', 'C': 'C (20%/100ms)'}
FILE_SIZE_MB = 10.0
COLORS = {'TCP': '#1f77b4', 'R-UDP/GBN': '#d62728'}

def load_jsonl(path):
    records = []
    if not os.path.exists(path):
        print(f'[AVISO] Não encontrado: {path}')
        return records
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                try: records.append(json.loads(line))
                except: pass
    return records

def build_df():
    tcp_recs  = load_jsonl(os.path.join(LOG_DIR, 'tcp_client_metrics.jsonl'))
    rudp_recs = load_jsonl(os.path.join(LOG_DIR, 'rudp_client_metrics.jsonl'))
    rows = []
    for proto, recs in [('TCP', tcp_recs), ('R-UDP/GBN', rudp_recs)]:
        for s in SCENARIOS:
            sub = [r for r in recs if r.get('scenario') == s]
            if not sub: continue
            thr   = [r.get('client_throughput_mbps') or r.get('throughput_mbps', 0) for r in sub]
            times = [r.get('client_elapsed_sec') or r.get('elapsed_sec', 0) for r in sub]
            retx  = [r.get('retransmissions', 0) for r in sub]
            rows.append({
                'protocol': proto, 'scenario': s,
                'scenario_label': SCENARIO_LABELS[s],
                'throughput_mean': np.mean(thr),
                'throughput_std':  np.std(thr, ddof=1) if len(thr)>1 else 0,
                'time_mean': np.mean(times),
                'time_std':  np.std(times, ddof=1) if len(times)>1 else 0,
                'retx_mean': np.mean(retx),
                'retx_std':  np.std(retx, ddof=1) if len(retx)>1 else 0,
                'n': len(sub)
            })
    return pd.DataFrame(rows)

df = build_df()
if df.empty:
    print('❌ Nenhuma métrica encontrada. Verifique o BASE_DIR.')
else:
    print('✅ Dados carregados:')
    display(df[['protocol','scenario','throughput_mean','throughput_std','time_mean','retx_mean','n']])

## Célula 4 — Throughput Médio (Plotly)

In [ ]:
fig = go.Figure()
for proto in ['TCP', 'R-UDP/GBN']:
    sub = df[df['protocol'] == proto]
    fig.add_trace(go.Bar(
        name=proto,
        x=sub['scenario_label'].tolist(),
        y=sub['throughput_mean'].tolist(),
        error_y=dict(type='data', array=sub['throughput_std'].tolist()),
        marker_color=COLORS[proto]
    ))
fig.update_layout(
    title='Throughput Médio por Cenário — TCP vs. R-UDP/GBN',
    xaxis_title='Cenário de Rede', yaxis_title='Throughput (Mbps)',
    barmode='group', template='plotly_white', legend_title='Protocolo'
)
fig.write_image(os.path.join(PLOT_DIR, 'throughput.png'), width=900, height=500)
fig.show()
print('✅ Salvo: throughput.png')

## Célula 5 — Throughput (Seaborn)

In [ ]:
fig2, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=df, x='scenario_label', y='throughput_mean',
            hue='protocol', errorbar=None, palette=COLORS, ax=ax)
ax.set_title('Throughput Médio — TCP vs. R-UDP/GBN (Seaborn)')
ax.set_xlabel('Cenário de Rede')
ax.set_ylabel('Throughput (Mbps)')
ax.legend(title='Protocolo')
plt.tight_layout()
fig2.savefig(os.path.join(PLOT_DIR, 'throughput_seaborn.png'), dpi=150)
plt.show()
print('✅ Salvo: throughput_seaborn.png')

## Célula 6 — Tempo de Transferência

In [ ]:
fig = go.Figure()
for proto in ['TCP', 'R-UDP/GBN']:
    sub = df[df['protocol'] == proto]
    fig.add_trace(go.Bar(
        name=proto,
        x=sub['scenario_label'].tolist(),
        y=sub['time_mean'].tolist(),
        error_y=dict(type='data', array=sub['time_std'].tolist()),
        marker_color=COLORS[proto]
    ))
fig.update_layout(
    title='Tempo Médio de Transferência por Cenário',
    xaxis_title='Cenário de Rede', yaxis_title='Tempo (s)',
    barmode='group', template='plotly_white'
)
fig.write_image(os.path.join(PLOT_DIR, 'transfer_time.png'), width=900, height=500)
fig.show()
print('✅ Salvo: transfer_time.png')

## Célula 7 — Retransmissões R-UDP/GBN

In [ ]:
sub = df[df['protocol'] == 'R-UDP/GBN']
fig = go.Figure(go.Bar(
    x=sub['scenario_label'].tolist(),
    y=sub['retx_mean'].tolist(),
    error_y=dict(type='data', array=sub['retx_std'].tolist()),
    marker_color='#d62728', name='Retransmissões'
))
fig.update_layout(
    title='Retransmissões Médias — R-UDP/GBN por Cenário',
    xaxis_title='Cenário de Rede', yaxis_title='Nº de Retransmissões',
    template='plotly_white'
)
fig.write_image(os.path.join(PLOT_DIR, 'retransmissions.png'), width=900, height=500)
fig.show()
print('✅ Salvo: retransmissions.png')

## Célula 8 — Validação Cruzada App vs. TCPDump

In [ ]:
def load_tcpdump_stats():
    stats = {s: {'TCP': [], 'RUDP': []} for s in SCENARIOS}
    for scenario in SCENARIOS:
        for key, label in [('tcp', 'TCP'), ('rudp', 'RUDP')]:
            pattern = os.path.join(CSV_DIR, f'scenario_{scenario}_{key}_rep*.csv')
            for f in sorted(glob.glob(pattern)):
                try:
                    df_csv = pd.read_csv(f)
                    if df_csv.empty: continue
                    stats[scenario][label].append({
                        'bytes': df_csv['length'].sum(),
                        'packets': len(df_csv)
                    })
                except Exception as e:
                    print(f'[aviso] {f}: {e}')
    return stats

tcpdump_stats = load_tcpdump_stats()
cv_rows = []
for proto_key, proto_label in [('TCP', 'TCP'), ('RUDP', 'R-UDP/GBN')]:
    for scenario in SCENARIOS:
        captures = tcpdump_stats[scenario][proto_key]
        if not captures: continue
        net_mb   = np.mean([c['bytes'] for c in captures]) / 1e6
        overhead = (net_mb - FILE_SIZE_MB) / FILE_SIZE_MB * 100
        cv_rows.append({
            'scenario': SCENARIO_LABELS[scenario], 'protocol': proto_label,
            'app_MB': FILE_SIZE_MB, 'net_MB': round(net_mb, 2),
            'overhead_pct': round(overhead, 2)
        })

if cv_rows:
    cv_df = pd.DataFrame(cv_rows)
    cv_df.to_csv(os.path.join(PLOT_DIR, 'cross_validation_table.csv'), index=False)
    print('Tabela de validação cruzada:')
    display(cv_df)

    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=['Volume: App vs TCPDump (MB)', 'Overhead de Rede (%)'])
    for proto in ['TCP', 'R-UDP/GBN']:
        sub = cv_df[cv_df['protocol'] == proto]
        color = COLORS[proto]
        fig.add_trace(go.Bar(name=f'{proto} App', x=sub['scenario'].tolist(),
                             y=sub['app_MB'].tolist(), marker_color=color,
                             opacity=0.4), row=1, col=1)
        fig.add_trace(go.Bar(name=f'{proto} TCPDump', x=sub['scenario'].tolist(),
                             y=sub['net_MB'].tolist(), marker_color=color,
                             opacity=1.0), row=1, col=1)
        fig.add_trace(go.Scatter(name=f'{proto} Overhead',
                                 x=sub['scenario'].tolist(), y=sub['overhead_pct'].tolist(),
                                 mode='lines+markers',
                                 marker=dict(color=color), line=dict(color=color)), row=1, col=2)
    fig.update_layout(title='Validação Cruzada: Aplicação vs. TCPDump',
                      barmode='group', template='plotly_white')
    fig.write_image(os.path.join(PLOT_DIR, 'cross_validation.png'), width=1100, height=500)
    fig.show()
    print('✅ Salvo: cross_validation.png')
else:
    print('[AVISO] Sem dados de tcpdump. Verifique CSV_DIR.')

## Célula 9 — Heatmap Comparativo

In [ ]:
pivot_thr  = df.pivot(index='protocol', columns='scenario', values='throughput_mean')
pivot_time = df.pivot(index='protocol', columns='scenario', values='time_mean')

fig_h, axes = plt.subplots(1, 2, figsize=(12, 3))
for ax, data, title in [
    (axes[0], pivot_thr,  'Throughput Médio (Mbps)'),
    (axes[1], pivot_time, 'Tempo Médio (s)'),
]:
    sns.heatmap(data, annot=True, fmt='.2f', cmap='YlOrRd', linewidths=0.5, ax=ax)
    ax.set_title(title)
    ax.set_xlabel('Cenário')
    ax.set_ylabel('Protocolo')

plt.suptitle('Comparação de Desempenho — TCP vs. R-UDP/GBN', fontsize=13)
plt.tight_layout()
fig_h.savefig(os.path.join(PLOT_DIR, 'heatmap.png'), dpi=150)
plt.show()
print('✅ Salvo: heatmap.png')

## Célula 10 — Resumo Estatístico Final

In [ ]:
print('=== RESUMO ESTATÍSTICO — PPGCC/UFPI ===')
print(f'Aluno: Arthur Sabino Santos | Matrícula: 20261005029')
print(f'Arquivo de teste: {FILE_SIZE_MB} MB | Protocolo GBN: W=8, T=0.3s')
print()
for scenario in SCENARIOS:
    print(f'--- Cenário {SCENARIO_LABELS[scenario]} ---')
    sub = df[df['scenario'] == scenario]
    for _, row in sub.iterrows():
        print(f"  {row['protocol']:12s} | "
              f"Throughput: {row['throughput_mean']:8.3f} ± {row['throughput_std']:.3f} Mbps | "
              f"Tempo: {row['time_mean']:8.2f} ± {row['time_std']:.2f} s | "
              f"Retx: {row['retx_mean']:.0f} | n={row['n']}")
    print()

print('=== GBN — Retransmissões esperadas pela fórmula ===')
W = 8
for p, label in [(0.0,'Cenário A'),(0.10,'Cenário B'),(0.20,'Cenário C')]:
    e = 1/(1-p)**W if p < 1 else float('inf')
    print(f'  {label} (p={p:.0%}): E[tx/pkt] = 1/(1-{p})^{W} = {e:.2f}')

print()
print(f'Gráficos em: {PLOT_DIR}')
for f in sorted(os.listdir(PLOT_DIR)):
    kb = os.path.getsize(os.path.join(PLOT_DIR,f))/1024
    print(f'  {f:<42} {kb:.1f} KB')